In [ ]:
import os
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, precision_score, recall_score,
    roc_auc_score
)
from xgboost import XGBClassifier

# ── CONFIG ──────────────────────────────────────────────────────────────────
DATA_DIR   = '/home/laura/Scriptie/bewerkte_data/THRawS'
THRESHOLDS = np.linspace(0.05, 0.95, 19)
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
X_train = np.load(os.path.join(DATA_DIR, 'X_train_THRawS.npy'))
y_train = np.load(os.path.join(DATA_DIR, 'y_train_THRawS.npy'))
X_test  = np.load(os.path.join(DATA_DIR, 'X_test_THRawS.npy'))
y_test  = np.load(os.path.join(DATA_DIR, 'y_test_THRawS.npy'))

In [ ]:
mc_model = DummyClassifier(strategy='prior', random_state=42)
mc_model.fit(X_train, y_train)

mc_pred = mc_model.predict(X_test)
mc_fire_idx = list(mc_model.classes_).index(1)
mc_probs = mc_model.predict_proba(X_test)[:, mc_fire_idx]

print("=== MAJORITY CLASS BASELINE ===")
print(f"Prior probability of fire (class 1): {mc_probs[0]:.4f}")
print(f"Accuracy: {accuracy_score(y_test, mc_pred):.3f}")
print(classification_report(y_test, mc_pred,
      target_names=['No event (0)', 'Event (1)'], zero_division=0))

## Vorige variant, zonder retraining en resampling

In [ ]:
# Random Forest
rf = RandomForestClassifier(
    n_estimators=100, random_state=42,
    n_jobs=-1, class_weight='balanced'
)
rf.fit(X_train, y_train)
fire_idx_rf = list(rf.classes_).index(1)
rf_probs = rf.predict_proba(X_test)[:, fire_idx_rf]

print("=== RANDOM FOREST (threshold 0.5) ===")
rf_pred_05 = (rf_probs >= 0.5).astype(int)
print(classification_report(y_test, rf_pred_05,
      target_names=['No event (0)', 'Event (1)'], zero_division=0))

# XGBoost
xgb_ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    random_state=42, n_jobs=-1,
    scale_pos_weight=xgb_ratio, eval_metric='logloss'
)
xgb.fit(X_train, y_train)
fire_idx_xgb = list(xgb.classes_).index(1)
xgb_probs = xgb.predict_proba(X_test)[:, fire_idx_xgb]

print("=== XGBOOST (threshold 0.5) ===")
xgb_pred_05 = (xgb_probs >= 0.5).astype(int)
print(classification_report(y_test, xgb_pred_05,
      target_names=['No event (0)', 'Event (1)'], zero_division=0))

In [ ]:
rf_results, xgb_results = [], []

for t in THRESHOLDS:
    rf_pred  = (rf_probs  >= t).astype(int)
    xgb_pred = (xgb_probs >= t).astype(int)

    rf_results.append({
        'threshold': t,
        'probs':     rf_probs,
        'preds':     rf_pred,
        'f1':        f1_score(y_test, rf_pred, zero_division=0),
        'f1_wtd':    f1_score(y_test, rf_pred, average='weighted', zero_division=0),
        'precision': precision_score(y_test, rf_pred, zero_division=0),
        'recall':    recall_score(y_test, rf_pred, zero_division=0),
        'accuracy':  accuracy_score(y_test, rf_pred),
        'auroc':     roc_auc_score(y_test, rf_probs),
    })
    xgb_results.append({
        'threshold': t,
        'probs':     xgb_probs,
        'preds':     xgb_pred,
        'f1':        f1_score(y_test, xgb_pred, zero_division=0),
        'f1_wtd':    f1_score(y_test, xgb_pred, average='weighted', zero_division=0),
        'precision': precision_score(y_test, xgb_pred, zero_division=0),
        'recall':    recall_score(y_test, xgb_pred, zero_division=0),
        'accuracy':  accuracy_score(y_test, xgb_pred),
        'auroc':     roc_auc_score(y_test, xgb_probs),
    })

print(f"Done — {len(THRESHOLDS)} thresholds evaluated.")

In [ ]:
for name, results in [('Random Forest', rf_results), ('XGBoost', xgb_results)]:
    best = max(results, key=lambda x: x['f1'])
    print(f"=== {name} ===")
    print(f"  Best threshold: {best['threshold']:.2f}")
    print(f"  F1:        {best['f1']:.3f}")
    print(f"  F1 wtd:    {best['f1_wtd']:.3f}")
    print(f"  Precision: {best['precision']:.3f}")
    print(f"  Recall:    {best['recall']:.3f}")
    print(f"  Accuracy:  {best['accuracy']:.3f}")
    print(f"  AUROC:     {best['auroc']:.3f}")
    print()

# Full table for Random Forest
print(f"{'Threshold':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Accuracy':>10}")
print("-" * 50)
for r in rf_results:
    print(f"  {r['threshold']:>8.2f} {r['f1']:>8.3f} {r['precision']:>10.3f} "
          f"{r['recall']:>8.3f} {r['accuracy']:>10.3f}")

In [ ]:
print(f"mc_model.classes_: {mc_model.classes_}")
print(f"Prior per klasse: {mc_model.predict_proba(X_test[:1])}")

## Cost-sensitive aanpak, maar dat is niet de goede manier

In [ ]:
rf_results_cs, xgb_results_cs = [], []

for t in THRESHOLDS:
    #pos_weight = np.clip((1 - t) / t, 0.1, 10)
    pos_weight = np.clip((1 - t) / t, 0.3, 3)

    # Random Forest
    rf_cs = RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1,
        class_weight={0: 1.0, 1: pos_weight}
    )
    rf_cs.fit(X_train, y_train)
    fire_idx = list(rf_cs.classes_).index(1)
    rf_prob_cs = rf_cs.predict_proba(X_test)[:, fire_idx]
    rf_pred_cs = (rf_prob_cs >= 0.5).astype(int)

    rf_results_cs.append({
        'threshold': t,
        'pos_weight': pos_weight,
        'f1':        f1_score(y_test, rf_pred_cs, zero_division=0),
        'precision': precision_score(y_test, rf_pred_cs, zero_division=0),
        'recall':    recall_score(y_test, rf_pred_cs, zero_division=0),
        'accuracy':  accuracy_score(y_test, rf_pred_cs),
        'auroc':     roc_auc_score(y_test, rf_prob_cs),
    })

    # XGBoost
    xgb_cs = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=-1,
        scale_pos_weight=pos_weight, eval_metric='logloss'
    )
    xgb_cs.fit(X_train, y_train)
    fire_idx_xgb = list(xgb_cs.classes_).index(1)
    xgb_prob_cs = xgb_cs.predict_proba(X_test)[:, fire_idx_xgb]
    xgb_pred_cs = (xgb_prob_cs >= 0.5).astype(int)

    xgb_results_cs.append({
        'threshold': t,
        'pos_weight': pos_weight,
        'f1':        f1_score(y_test, xgb_pred_cs, zero_division=0),
        'precision': precision_score(y_test, xgb_pred_cs, zero_division=0),
        'recall':    recall_score(y_test, xgb_pred_cs, zero_division=0),
        'accuracy':  accuracy_score(y_test, xgb_pred_cs),
        'auroc':     roc_auc_score(y_test, xgb_prob_cs),
    })

print(f"{'Threshold':>10} {'PosWeight':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Accuracy':>10}")
print("-" * 65)
for r in rf_results_cs:
    print(f"  {r['threshold']:>8.2f} {r['pos_weight']:>10.3f} {r['f1']:>8.3f} "
          f"{r['precision']:>10.3f} {r['recall']:>8.3f} {r['accuracy']:>10.3f}")

## Best approach, retrain and resample

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RESAMPLING PER THRESHOLD — Random Forest
# ════════════════════════════════════════════════════════════════════════
#
# Doel: simuleer 19 "verschillende modellen" door voor elke threshold een
# eigen trainingsdataset samen te stellen, in plaats van één model te
# trainen en achteraf de afkapwaarde te verschuiven.
#
# Hoe: we passen de VERHOUDING brand/geen-brand in de trainingsdata aan.
# Een model dat op veel brand-voorbeelden trainde, leert sneller "brand"
# te roepen — dat simuleert een model dat bedoeld is voor een lage
# threshold. Een model met weinig brand-voorbeelden wordt voorzichtiger —
# dat simuleert een hoge threshold.
#
# Belangrijk verschil met class_weight: class_weight verandert alleen hoe
# zwaar een sample meetelt bij het BEREKENEN van een splitsing in de boom.
# Resampling verandert welke RIJEN het model daadwerkelijk ziet tijdens
# training. Dat laatste is wat "een andere dataset per threshold" betekent.

# Indices van de twee klassen in de ORIGINELE trainingsset (vast, verandert niet)
base_idx_0 = np.where(y_train == 0)[0]  # geen brand — blijft altijd hetzelfde
base_idx_1 = np.where(y_train == 1)[0]  # brand — wordt per threshold aangepast
n0, n1 = len(base_idx_0), len(base_idx_1)

rf_results_resample = []

for t in THRESHOLDS:
    # Stap 1: bereken hoeveel brand-samples we willen, gebaseerd op de threshold.
    # Dezelfde formule als bij class_weight, maar nu gebruikt als resample-factor
    # in plaats van als gewicht.
    #   - lage t  → grote pos_weight → veel méér brand-samples (oversampling)
    #   - hoge t  → kleine pos_weight → veel minder brand-samples (undersampling)
    pos_weight = (1 - t) / t
    target_n1 = max(int(round(n1 * pos_weight)), 5)  # minimaal 5 samples, anders te instabiel

    # Stap 2: trek een nieuwe selectie brand-samples uit de ORIGINELE brand-samples.
    # Met teruglegging (replace=True) als we er meer nodig hebben dan er zijn
    # (dupliceren), zonder teruglegging als we er minder nodig hebben (weglaten).
    if target_n1 >= n1:
        sampled_idx_1 = np.random.choice(base_idx_1, size=target_n1, replace=True)
    else:
        sampled_idx_1 = np.random.choice(base_idx_1, size=target_n1, replace=False)

    # Stap 3: zet de nieuwe trainingsset in elkaar — alle geen-brand samples
    # plus de (aangepaste hoeveelheid) brand samples, geshuffeld zodat de
    # volgorde niet de klasse verraadt.
    resampled_idx = np.concatenate([base_idx_0, sampled_idx_1])
    np.random.shuffle(resampled_idx)

    X_train_t = X_train[resampled_idx]
    y_train_t = y_train[resampled_idx]

    # Stap 4: train een VERS model op deze aangepaste data.
    # Geen class_weight hier — de aanpassing zit al in de data zelf,
    # dubbel toepassen zou het effect onnodig versterken.
    rf_t = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_t.fit(X_train_t, y_train_t)

    fire_idx = list(rf_t.classes_).index(1)
    probs_t = rf_t.predict_proba(X_test)[:, fire_idx]

    # Stap 5: evalueer altijd op de NEUTRALE threshold 0.5.
    # De threshold-simulatie is al verwerkt in de trainingsdata (stap 1-3),
    # dus de afkapwaarde bij voorspellen moet neutraal blijven.
    pred_t = (probs_t >= 0.5).astype(int)

    rf_results_resample.append({
        'threshold':    t,             # de gesimuleerde threshold
        'n1_resampled': target_n1,     # ter controle: hoeveel brand-samples gebruikt
        'f1':        f1_score(y_test, pred_t, zero_division=0),
        'precision': precision_score(y_test, pred_t, zero_division=0),
        'recall':    recall_score(y_test, pred_t, zero_division=0),
        'accuracy':  accuracy_score(y_test, pred_t),
        'auroc':     roc_auc_score(y_test, probs_t),
    })

print(f"{'Threshold':>10} {'N(brand)':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Accuracy':>10}")
print("-" * 60)
for r in rf_results_resample:
    print(f"  {r['threshold']:>8.2f} {r['n1_resampled']:>10} {r['f1']:>8.3f} "
          f"{r['precision']:>10.3f} {r['recall']:>8.3f} {r['accuracy']:>10.3f}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RESAMPLING PER THRESHOLD — XGBoost (zelfde principe als Random Forest)
# ════════════════════════════════════════════════════════════════════════

xgb_results_resample = []

for t in THRESHOLDS:
    pos_weight = (1 - t) / t
    target_n1 = max(int(round(n1 * pos_weight)), 5)

    if target_n1 >= n1:
        sampled_idx_1 = np.random.choice(base_idx_1, size=target_n1, replace=True)
    else:
        sampled_idx_1 = np.random.choice(base_idx_1, size=target_n1, replace=False)

    resampled_idx = np.concatenate([base_idx_0, sampled_idx_1])
    np.random.shuffle(resampled_idx)

    X_train_t = X_train[resampled_idx]
    y_train_t = y_train[resampled_idx]

    # Geen scale_pos_weight — zelfde reden als bij Random Forest.
    xgb_t = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=-1, eval_metric='logloss'
    )
    xgb_t.fit(X_train_t, y_train_t)

    fire_idx = list(xgb_t.classes_).index(1)
    probs_t = xgb_t.predict_proba(X_test)[:, fire_idx]
    pred_t  = (probs_t >= 0.5).astype(int)  # neutrale evaluatie-threshold

    xgb_results_resample.append({
        'threshold':    t,
        'n1_resampled': target_n1,
        'f1':        f1_score(y_test, pred_t, zero_division=0),
        'precision': precision_score(y_test, pred_t, zero_division=0),
        'recall':    recall_score(y_test, pred_t, zero_division=0),
        'accuracy':  accuracy_score(y_test, pred_t),
        'auroc':     roc_auc_score(y_test, probs_t),
    })

print(f"{'Threshold':>10} {'N(brand)':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Accuracy':>10}")
print("-" * 60)
for r in xgb_results_resample:
    print(f"  {r['threshold']:>8.2f} {r['n1_resampled']:>10} {r['f1']:>8.3f} "
          f"{r['precision']:>10.3f} {r['recall']:>8.3f} {r['accuracy']:>10.3f}")

In [ ]:
import matplotlib.pyplot as plt

thresholds_plot = [r['threshold'] for r in rf_results_resample]
rf_f1   = [r['f1']   for r in rf_results_resample]
xgb_f1  = [r['f1']   for r in xgb_results_resample]
rf_auc  = [r['auroc'] for r in rf_results_resample]
xgb_auc = [r['auroc'] for r in xgb_results_resample]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# F1 vergelijking
axes[0].plot(thresholds_plot, rf_f1,  marker='o', label='Random Forest', linewidth=2, color='royalblue')
axes[0].plot(thresholds_plot, xgb_f1, marker='s', label='XGBoost',       linewidth=2, color='darkorange')
axes[0].axvline(0.5, color='gray', linestyle=':', label='Threshold 0.5')
axes[0].set_title('F1 Score', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Threshold (gesimuleerd via resampling)')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.05)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUROC vergelijking
axes[1].plot(thresholds_plot, rf_auc,  marker='o', label='Random Forest', linewidth=2, color='royalblue')
axes[1].plot(thresholds_plot, xgb_auc, marker='s', label='XGBoost',       linewidth=2, color='darkorange')
axes[1].axvline(0.5, color='gray', linestyle=':', label='Threshold 0.5')
axes[1].set_title('AUROC', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Threshold (gesimuleerd via resampling)')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle('THRawS: Resampling per Threshold — RF vs XGBoost', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Save sweep results

In [ ]:
# Ground truth
np.save(os.path.join(DATA_DIR, 'y_test_THRawS.npy'), y_test)

# Majority class probabilities (threshold-independent)
np.save(os.path.join(DATA_DIR, 'y_prob_majority_THRawS.npy'), mc_probs)

# Per-threshold results
np.save(os.path.join(DATA_DIR, 'results_rf_THRawS.npy'),  rf_results,  allow_pickle=True)
np.save(os.path.join(DATA_DIR, 'results_xgb_THRawS.npy'), xgb_results, allow_pickle=True)

# Probs at threshold 0.5 for the evaluation notebook
np.save(os.path.join(DATA_DIR, 'y_prob_rf_THRawS.npy'),  rf_probs)
np.save(os.path.join(DATA_DIR, 'y_prob_xgb_THRawS.npy'), xgb_probs)

print("Saved all results to disk.")

## Save resampling results

In [ ]:
# Sla de resampling-resultaten apart op, met een andere naam dan de
# gewone threshold sweep, zodat beide aanpakken vergelijkbaar blijven.
np.save(os.path.join(DATA_DIR, 'results_rf_resample_THRawS.npy'),  rf_results_resample,  allow_pickle=True)
np.save(os.path.join(DATA_DIR, 'results_xgb_resample_THRawS.npy'), xgb_results_resample, allow_pickle=True)

print("Saved resampling results:")
print(f"  results_rf_resample_THRawS.npy  — {len(rf_results_resample)} thresholds")
print(f"  results_xgb_resample_THRawS.npy — {len(xgb_results_resample)} thresholds")